# SATD Classification on New Issue Data — Optimized (2x T4)

Optimized rewrite of `satd-classifications-on-new-issue-data.ipynb`. Same task, same models,
same output schema — but built to finish inside a Kaggle free-tier session.

**Task:** run the trained Pipeline A models on new issue data, 3 variations:
1. Title only
2. Description only
3. Title + Description

**Models:** Qwen3-Embedding-0.6B (frozen) + XGBoost identification and categorization heads.

**Output:** Parquet with `ID, Title, Description, Title_SATD, Description_SATD, Title_Description_SATD`.

## What changed vs the original notebook

| Change | Why |
|---|---|
| Both T4 GPUs used, one worker process per GPU | The original used a single GPU; this is close to a 2x speedup |
| FP16 weights | Roughly 1.5-2x faster on T4 tensor cores |
| `max_seq_length` cap (default 256 tokens) | Long issue descriptions dominated the runtime; attention cost grows with length |
| Length-sorted batching | Cuts padding waste, the single biggest win on mixed-length text |
| Embeddings classified per chunk and discarded | Avoids holding 3 x 458k x 1024 float arrays in RAM |
| Per-shard checkpoints written to disk | A crash or timeout no longer loses the whole run |
| Built-in benchmark cell | Measures real throughput on a sample and projects total runtime before you commit |

Pooling, prompt handling and `normalize_embeddings=True` are kept identical to the original so
predictions stay comparable.

## Kaggle setup
1. Settings -> Accelerator -> **GPU T4 x2**
2. Settings -> Internet **on** (first run downloads the encoder from HuggingFace)
3. Add `issue_202608272234.parquet` as a dataset
4. Add the `models/` folder as a dataset
5. Update `INPUT_PATH` and `MODEL_DIR` in the config cell

In [13]:
!ls -R /kaggle/input | head -60

/kaggle/input:
datasets

/kaggle/input/datasets:
shahriarpias

/kaggle/input/datasets/shahriarpias:
satd-issue-parquet
satd-pipeline-a-models

/kaggle/input/datasets/shahriarpias/satd-issue-parquet:
issue_202608272234.parquet

/kaggle/input/datasets/shahriarpias/satd-pipeline-a-models:
04_trained_models

/kaggle/input/datasets/shahriarpias/satd-pipeline-a-models/04_trained_models:
categorization_code_comment_e5_logreg.joblib
categorization_code_comment_e5_xgboost.joblib
categorization_code_comment_qwen3_logreg.joblib
categorization_code_comment_qwen3_xgboost.joblib
categorization_commit_e5_logreg.joblib
categorization_commit_e5_xgboost.joblib
categorization_commit_qwen3_logreg.joblib
categorization_commit_qwen3_xgboost.joblib
categorization_issue_e5_logreg.joblib
categorization_issue_e5_xgboost.joblib
categorization_issue_qwen3_logreg.joblib
categorization_issue_qwen3_xgboost.joblib
categorization_pull_request_e5_logreg.joblib
categorization_pull_request_e5_xgboost.joblib
categorizatio

In [14]:
!pip install -q pandas pyarrow sentence-transformers xgboost torch

## 1. Configuration

In [15]:
import os
import sys
import json
import time
import subprocess
import numpy as np
import pandas as pd
import torch
import warnings
warnings.filterwarnings('ignore')

# === KAGGLE PATHS — UPDATE THESE ===
INPUT_PATH = "/kaggle/input/datasets/shahriarpias/satd-issue-parquet/issue_202608272234.parquet"
MODEL_DIR  = "/kaggle/input/datasets/shahriarpias/satd-pipeline-a-models/04_trained_models"

OUTPUT_DIR  = "/kaggle/working"
SHARD_DIR   = os.path.join(OUTPUT_DIR, "shards")
LOG_DIR     = os.path.join(OUTPUT_DIR, "logs")
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "satd_classification_results.parquet")
WORKER_PATH = os.path.join(OUTPUT_DIR, "satd_worker.py")
CONFIG_PATH = os.path.join(OUTPUT_DIR, "satd_worker_config.json")

IDENTIFICATION_MODEL = os.path.join(MODEL_DIR, "identification_issue_qwen3_xgboost.joblib")
CATEGORIZATION_MODEL = os.path.join(MODEL_DIR, "categorization_issue_qwen3_xgboost.joblib")

# Shared HF cache so both worker processes reuse one download
HF_CACHE = os.path.join(OUTPUT_DIR, "hf_cache")
os.environ["HF_HOME"] = HF_CACHE

# === PERFORMANCE KNOBS ===
EMBED_MODEL    = "Qwen/Qwen3-Embedding-0.6B"
MAX_SEQ_LENGTH = 256   # token cap. 512 is more faithful but roughly 2x slower on descriptions
BATCH_SIZE     = 128   # lower to 64 if you hit CUDA OOM
USE_FP16       = True

for d in (SHARD_DIR, LOG_DIR, HF_CACHE):
    os.makedirs(d, exist_ok=True)

NUM_GPUS = torch.cuda.device_count()
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPUs detected  : {NUM_GPUS}")
for i in range(NUM_GPUS):
    print(f"  [{i}] {torch.cuda.get_device_name(i)}")
if NUM_GPUS < 2:
    print("\nWARNING: fewer than 2 GPUs. Set Accelerator to 'GPU T4 x2' for the full speedup.")

print(f"\nInput  : {INPUT_PATH}")
print(f"Models : {MODEL_DIR}")
print(f"Output : {OUTPUT_PATH}")

CUDA available : True
GPUs detected  : 2
  [0] Tesla T4
  [1] Tesla T4

Input  : /kaggle/input/datasets/shahriarpias/satd-issue-parquet/issue_202608272234.parquet
Models : /kaggle/input/datasets/shahriarpias/satd-pipeline-a-models/04_trained_models
Output : /kaggle/working/satd_classification_results.parquet


## 2. Inspect the data

Text length is the main driver of runtime, so we look at it before committing to a full run.

In [16]:
df = pd.read_parquet(INPUT_PATH, columns=["ID", "Title", "Description"])
print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")

print("\nMissing values:")
print(df.isnull().sum())

title_len = df["Title"].fillna("").str.len()
desc_len  = df["Description"].fillna("").str.len()
print("\nCharacter length (a rough token proxy: ~4 chars per token):")
print(pd.DataFrame({"Title": title_len.describe(), "Description": desc_len.describe()}))
print(f"\nDescriptions longer than {MAX_SEQ_LENGTH * 4} chars (will be truncated): "
      f"{(desc_len > MAX_SEQ_LENGTH * 4).mean():.1%}")

df.head()

Loaded 458232 rows
Columns: ['ID', 'Title', 'Description']

Missing values:
ID                 0
Title              0
Description    29128
dtype: int64

Character length (a rough token proxy: ~4 chars per token):
               Title   Description
count  458232.000000  4.582320e+05
mean       58.431642  9.292859e+02
std        24.893745  4.701393e+03
min         1.000000  0.000000e+00
25%        41.000000  1.930000e+02
50%        55.000000  4.090000e+02
75%        72.000000  7.580000e+02
max       269.000000  1.391304e+06

Descriptions longer than 1024 chars (will be truncated): 16.3%


,ID,Title,Description
0,912,Create a Java client for Receptor,"As a developer, I'd like to create a [java cli..."
1,913,Create Boot based ModuleRunner,"As a developer, I'd like to build isolated Boo..."
2,914,Create a pluggable runtime SPI,"As a developer, I'd like to migrate module dep..."
3,915,Simplify GPDB UX around parameters for the Sqo...,"As a developer, I'd like to have a simplified ..."
4,916,Sqoop Module not running,Can not get any jobs created using Scoop Modu...


## 3. Pre-download the encoder

Done once in the main process so the two workers do not race on the same cache.

In [17]:
from huggingface_hub import snapshot_download

t0 = time.time()
local_path = snapshot_download(EMBED_MODEL)
print(f"Encoder cached at {local_path} ({time.time() - t0:.0f}s)")

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Encoder cached at /kaggle/working/hf_cache/hub/models--Qwen--Qwen3-Embedding-0.6B/snapshots/97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3 (0s)


## 4. Worker script

One process per GPU. Each worker takes an interleaved shard of the rows (`rows[shard::n_shards]`),
embeds and classifies all three text variations, and writes its own parquet checkpoint.

Interleaving rather than contiguous slicing keeps the shards length-balanced, so both GPUs finish
at roughly the same time.

In [18]:
%%writefile /kaggle/working/satd_worker.py
"""Embed + classify one shard of the issue data on a single GPU.

Usage: python satd_worker.py <shard_index> <num_shards> <config.json>
"""
import json
import os
import sys
import time

import joblib
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

SHARD = int(sys.argv[1])
NUM_SHARDS = int(sys.argv[2])
CFG = json.load(open(sys.argv[3]))


def log(msg):
    print(f"[shard {SHARD}] {msg}", flush=True)


# ---------------------------------------------------------------- load data
df = pd.read_parquet(CFG["input_path"], columns=["ID", "Title", "Description"])
if CFG.get("row_limit"):
    df = df.iloc[: CFG["row_limit"]]
df["Title"] = df["Title"].fillna("")
df["Description"] = df["Description"].fillna("")

row_pos = np.arange(len(df))[SHARD::NUM_SHARDS]
shard_df = df.iloc[row_pos].reset_index(drop=True)
del df
log(f"{len(shard_df)} rows")

# ------------------------------------------------------------- load encoder
model_kwargs = {"torch_dtype": torch.float16} if CFG["use_fp16"] else {}
embedder = SentenceTransformer(
    CFG["embed_model"],
    device="cuda",
    model_kwargs=model_kwargs,
)
embedder.max_seq_length = CFG["max_seq_length"]
embedder.eval()
log(f"encoder ready on {torch.cuda.get_device_name(0)} "
    f"(fp16={CFG['use_fp16']}, max_seq_length={CFG['max_seq_length']})")

identification_model = joblib.load(CFG["identification_model"])
categorization_model = joblib.load(CFG["categorization_model"])
log(f"identification classes: {identification_model.classes_}")
log(f"categorization classes: {categorization_model.classes_}")


# ------------------------------------------------------------- classification
def classify(embeddings):
    """Identification first; categorize only the rows predicted SATD."""
    preds = np.asarray(identification_model.predict(embeddings), dtype=object)
    satd_mask = preds == "SATD"
    if satd_mask.sum():
        preds[satd_mask] = np.asarray(
            categorization_model.predict(embeddings[satd_mask]), dtype=object
        )
    return preds


def embed_and_classify(texts, label):
    """Length-sorted batching, classify each batch immediately, keep only labels."""
    n = len(texts)
    batch_size = CFG["batch_size"]
    # Sort by length so each batch pads to a similar width instead of to the longest text.
    order = np.argsort([len(t) for t in texts], kind="stable")
    out = np.empty(n, dtype=object)

    start = time.time()
    n_batches = (n + batch_size - 1) // batch_size
    for bi, s in enumerate(range(0, n, batch_size)):
        idx = order[s : s + batch_size]
        batch = [texts[i] for i in idx]
        with torch.inference_mode():
            emb = embedder.encode(
                batch,
                batch_size=len(batch),
                normalize_embeddings=True,
                convert_to_numpy=True,
                show_progress_bar=False,
            )
        out[idx] = classify(np.asarray(emb, dtype=np.float32))

        if bi % 50 == 0 or bi == n_batches - 1:
            done = min(s + batch_size, n)
            elapsed = time.time() - start
            rate = done / max(elapsed, 1e-9)
            eta = (n - done) / max(rate, 1e-9)
            log(f"{label}: {done}/{n} ({done / n:.1%}) "
                f"{rate:.0f} rows/s elapsed={elapsed / 60:.1f}m eta={eta / 60:.1f}m")

    log(f"{label}: done in {(time.time() - start) / 60:.1f}m")
    return out


# ------------------------------------------------------------------ run
titles = shard_df["Title"].tolist()
descriptions = shard_df["Description"].tolist()
combined = (shard_df["Title"] + " " + shard_df["Description"]).tolist()

wall = time.time()
result = pd.DataFrame({
    "row_pos": row_pos,
    "ID": shard_df["ID"].values,
    "Title_SATD": embed_and_classify(titles, "title"),
    "Description_SATD": embed_and_classify(descriptions, "description"),
    "Title_Description_SATD": embed_and_classify(combined, "title+description"),
})

out_path = os.path.join(CFG["shard_dir"], f"shard_{SHARD}_of_{NUM_SHARDS}.parquet")
result.to_parquet(out_path, index=False)
log(f"wrote {out_path} in {(time.time() - wall) / 60:.1f}m total")

Overwriting /kaggle/working/satd_worker.py


In [19]:
def write_config(row_limit=None, shard_dir=SHARD_DIR):
    cfg = {
        "input_path": INPUT_PATH,
        "identification_model": IDENTIFICATION_MODEL,
        "categorization_model": CATEGORIZATION_MODEL,
        "shard_dir": shard_dir,
        "embed_model": EMBED_MODEL,
        "max_seq_length": MAX_SEQ_LENGTH,
        "batch_size": BATCH_SIZE,
        "use_fp16": USE_FP16,
        "row_limit": row_limit,
    }
    os.makedirs(shard_dir, exist_ok=True)
    with open(CONFIG_PATH, "w") as f:
        json.dump(cfg, f, indent=2)
    return cfg


def run_workers(num_shards, tag="run"):
    """Launch one worker per GPU and stream their logs until all finish."""
    procs = []
    for shard in range(num_shards):
        env = dict(os.environ)
        env["CUDA_VISIBLE_DEVICES"] = str(shard % max(NUM_GPUS, 1))
        env["HF_HOME"] = HF_CACHE
        env["TOKENIZERS_PARALLELISM"] = "false"
        log_path = os.path.join(LOG_DIR, f"{tag}_shard_{shard}.log")
        log_file = open(log_path, "w")
        proc = subprocess.Popen(
            [sys.executable, WORKER_PATH, str(shard), str(num_shards), CONFIG_PATH],
            env=env, stdout=log_file, stderr=subprocess.STDOUT,
        )
        procs.append((shard, proc, log_file, log_path))
        print(f"launched shard {shard} on GPU {env['CUDA_VISIBLE_DEVICES']} -> {log_path}")

    seen = {shard: 0 for shard, *_ in procs}
    while any(p.poll() is None for _, p, _, _ in procs):
        time.sleep(30)
        for shard, _, _, log_path in procs:
            with open(log_path) as f:
                lines = f.readlines()
            for line in lines[seen[shard]:]:
                print(line.rstrip())
            seen[shard] = len(lines)

    for shard, proc, log_file, log_path in procs:
        log_file.close()
        with open(log_path) as f:
            lines = f.readlines()
        for line in lines[seen[shard]:]:
            print(line.rstrip())
        print(f"shard {shard} exited with code {proc.returncode}")

    return all(p.returncode == 0 for _, p, _, _ in procs)


print("helpers ready")

helpers ready


## 5. Benchmark (recommended)

Runs the full worker on a small sample and projects the total runtime. Takes a few minutes and
tells you whether the current `MAX_SEQ_LENGTH` / `BATCH_SIZE` settings fit your time budget.
Skip it if you already know your numbers.

In [20]:
BENCH_ROWS = 4000
BENCH_DIR = os.path.join(OUTPUT_DIR, "shards_bench")

total_rows = len(pd.read_parquet(INPUT_PATH, columns=["ID"]))
n_workers = max(NUM_GPUS, 1)

write_config(row_limit=BENCH_ROWS, shard_dir=BENCH_DIR)
t0 = time.time()
ok = run_workers(n_workers, tag="bench")
bench_seconds = time.time() - t0

if ok:
    # Model load is a fixed cost (~1-2 min); subtract a conservative estimate before extrapolating.
    load_overhead = 90
    per_row = max(bench_seconds - load_overhead, 1) / BENCH_ROWS
    projected = per_row * total_rows + load_overhead
    print(f"\nBenchmark: {BENCH_ROWS} rows in {bench_seconds / 60:.1f}m on {n_workers} GPU(s)")
    print(f"Projected full run ({total_rows} rows): {projected / 3600:.1f} hours")
    if projected > 3 * 3600:
        print("\nOver 3 hours. To go faster: lower MAX_SEQ_LENGTH (256 -> 192 or 128),"
              " or raise BATCH_SIZE if GPU memory allows.")
else:
    print("Benchmark failed — check the logs above before running the full job.")

launched shard 0 on GPU 0 -> /kaggle/working/logs/bench_shard_0.log
launched shard 1 on GPU 1 -> /kaggle/working/logs/bench_shard_1.log
[shard 0] 2000 rows

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 353.67it/s, Materializing param=norm.weight]
[shard 0] encoder ready on Tesla T4 (fp16=True, max_seq_length=256)
[shard 0] identification classes: [0 1]
[shard 0] categorization classes: [0 1 2 3]
[shard 0] title: 128/2000 (6.4%) 110 rows/s elapsed=0.0m eta=0.3m
[shard 0] title: 2000/2000 (100.0%) 440 rows/s elapsed=0.1m eta=0.0m
[shard 0] title: done in 0.1m
[shard 0] description: 128/2000 (6.4%) 1252 rows/s elapsed=0.0m eta=0.0m
[shard 1] 2000 rows

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 442.21it/s, Materializing param=norm.weight]
[shard 1] encoder ready on Tesla T4 (fp16=True, max_seq_length=256)
[shard 1] identification classes: [0 1]
[shard 1] categorization classes: [0 1 2 3]
[shard 1] title: 128/2000 (6.4%) 118 rows/s elapsed=0.0m eta=0.3m
[shard 1] ti

## 6. Full run

Both GPUs work in parallel. Progress from each shard is printed every 30 seconds.

In [21]:
n_workers = max(NUM_GPUS, 1)
write_config(row_limit=None, shard_dir=SHARD_DIR)

t0 = time.time()
ok = run_workers(n_workers, tag="full")
print(f"\nAll workers finished in {(time.time() - t0) / 3600:.2f} hours (success={ok})")

launched shard 0 on GPU 0 -> /kaggle/working/logs/full_shard_0.log
launched shard 1 on GPU 1 -> /kaggle/working/logs/full_shard_1.log
[shard 0] 229116 rows

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 319.95it/s, Materializing param=norm.weight]
[shard 0] encoder ready on Tesla T4 (fp16=True, max_seq_length=256)
[shard 0] identification classes: [0 1]
[shard 0] categorization classes: [0 1 2 3]
[shard 0] title: 128/229116 (0.1%) 222 rows/s elapsed=0.0m eta=17.2m
[shard 0] title: 6528/229116 (2.8%) 933 rows/s elapsed=0.1m eta=4.0m
[shard 1] 229116 rows

Loading weights: 100%|██████████| 310/310 [00:01<00:00, 272.29it/s, Materializing param=norm.weight]
[shard 1] encoder ready on Tesla T4 (fp16=True, max_seq_length=256)
[shard 1] identification classes: [0 1]
[shard 1] categorization classes: [0 1 2 3]
[shard 1] title: 128/229116 (0.1%) 225 rows/s elapsed=0.0m eta=16.9m
[shard 1] title: 6528/229116 (2.8%) 952 rows/s elapsed=0.1m eta=3.9m
[shard 0] title: 12928/229116 (5.6%) 8

## 7. Merge shards and save

In [22]:
import glob

shard_files = sorted(glob.glob(os.path.join(SHARD_DIR, "shard_*_of_*.parquet")))
print(f"Found {len(shard_files)} shard files in {SHARD_DIR}")

if not shard_files:
    # Nothing was written: section 6 never ran, is still running, or the workers died.
    bench_files = glob.glob(os.path.join(OUTPUT_DIR, "shards_bench", "shard_*_of_*.parquet"))
    log_files = sorted(glob.glob(os.path.join(LOG_DIR, "*.log")))
    print("\nNo shards to merge. Diagnostics:")
    print(f"  benchmark shards present : {len(bench_files)} "
          f"(these are the 4k-row sample, not the full run)")
    print(f"  worker logs found        : {[os.path.basename(p) for p in log_files]}")
    for p in log_files:
        with open(p) as f:
            tail = f.readlines()[-15:]
        print(f"\n--- tail of {os.path.basename(p)} ---")
        print("".join(tail).rstrip() or "(empty)")
    raise SystemExit(
        "No shard files. Run section 6 (full run) first; if you already did, "
        "read the worker log tails above for the real error."
    )

preds = pd.concat([pd.read_parquet(p) for p in shard_files], ignore_index=True)
preds = preds.sort_values("row_pos").reset_index(drop=True)

df = pd.read_parquet(INPUT_PATH, columns=["ID", "Title", "Description"])
df["Title"] = df["Title"].fillna("")
df["Description"] = df["Description"].fillna("")

# Partial results are still worth keeping, so report coverage instead of asserting blindly.
missing = len(df) - len(preds)
if missing:
    print(f"\nWARNING: {missing} of {len(df)} rows have no prediction "
          f"({len(preds) / len(df):.1%} complete). Saving a partial result.")
    print("Unfinished shards: check the logs, then re-run section 6 for the missing shard.")

assert preds["row_pos"].is_unique, "duplicate row_pos across shards — stale shard files present?"
assert preds["row_pos"].max() < len(df), "row_pos out of range — shards came from a different input"
assert (preds["ID"].values == df["ID"].values[preds["row_pos"].values]).all(), \
    "ID misalignment between shards and input"

for col in ["Title_SATD", "Description_SATD", "Title_Description_SATD"]:
    df[col] = pd.NA
    df.loc[preds["row_pos"].values, col] = preds[col].values

df.to_parquet(OUTPUT_PATH, index=False)
print(f"\nResults saved to: {OUTPUT_PATH}")
print(f"Total rows: {len(df)}, classified: {df['Title_SATD'].notna().sum()}")

Found 2 shard files in /kaggle/working/shards

Results saved to: /kaggle/working/satd_classification_results.parquet
Total rows: 458232, classified: 458232


In [23]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)
for col in ["Title_SATD", "Description_SATD", "Title_Description_SATD"]:
    print(f"\n{col}:")
    print(df[col].value_counts())
    satd = (df[col] != "Not-SATD").sum()
    print(f"  SATD total: {satd} ({satd / len(df):.1%})")

SUMMARY

Title_SATD:
Title_SATD
0    283019
1    175213
Name: count, dtype: int64
  SATD total: 458232 (100.0%)

Description_SATD:
Description_SATD
1    269814
0    188418
Name: count, dtype: int64
  SATD total: 458232 (100.0%)

Title_Description_SATD:
Title_Description_SATD
1    252778
0    205454
Name: count, dtype: int64
  SATD total: 458232 (100.0%)


In [24]:
df.head(10)

,ID,Title,Description,Title_SATD,Description_SATD,Title_Description_SATD
0,912,Create a Java client for Receptor,"As a developer, I'd like to create a [java cli...",0,1,0
1,913,Create Boot based ModuleRunner,"As a developer, I'd like to build isolated Boo...",0,1,1
2,914,Create a pluggable runtime SPI,"As a developer, I'd like to migrate module dep...",1,1,1
3,915,Simplify GPDB UX around parameters for the Sqo...,"As a developer, I'd like to have a simplified ...",0,0,1
4,916,Sqoop Module not running,Can not get any jobs created using Scoop Modu...,0,1,1
5,917,Improve performance of TupleBuilder,"As a developer, I'd like to bench test cases a...",1,1,1
6,918,Revisit benchmark matrix for Sqoop vs. jdbshdf...,"As a developer, I'd like to revisit performanc...",1,1,1
7,919,Produce Kafka Baseline numbers on Rackspace,,0,0,0
8,920,Acceptance Tests needs to wait for JobDefiniti...,After the Introduction to XD-2861 the acquisit...,1,1,1
9,921,Create a reference architecture for high throu...,"As a developer, I'd like to have the XD + Kafk...",0,0,0
